# Reference-based spatial deconvolution

Estimate contributions from named cell types using your target counts and fixed reference profiles.
Required files: `counts.h5ad` and `reference.h5ad`. Reference rows are cell types and columns are genes;
values are mean raw counts per cell type, before row normalization. Prepare your reference marker
panel before running this notebook. Target coordinates go in `obsm["spatial"]`.

The reference remains fixed during fitting. Dividing molecular weights by type-specific reference
signal yields produces estimated cell fractions, assuming those relative yields represent the target.
This example uses the CPU fixed-reference solver with spatial weight zero.

Study data sources: [Xenium lymph node](https://www.10xgenomics.com/datasets/preview-data-xenium-prime-gene-expression)
and the [Flex reference release](https://www.10xgenomics.com/datasets/320k_scFFPE_16-plex_GEM-X_FLEX).
[Study illustration](https://dsong-lab.github.io/spDecon/_static/figures/xenium_maps.png).


## 1. Configure your input

Install the notebook dependencies, then edit `DATA_DIR`, the input filenames and `K` below.
Supply your own `.h5ad` files with unique spot and feature IDs. RNA fitting uses raw counts;
set `COUNTS_LAYER="counts"` when they are stored in a layer. Protein inputs should contain
your prepared nonnegative assay measurements. Each fitted spot needs positive total signal.

Running the fitting cells trains a new model. `DEVICE` accepts `"auto"`, `"cpu"` or `"cuda"`.
The study's reference-free and joint analyses used GPUs. Inspect the returned diagnostics
to determine whether your fit converged. Large inputs require enough memory for dense matrices.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
from IPython.display import display
from tutorials.common import correlation, align, spatial, pies, legend, save_figure, save_table
from spDecon.reproducibility import set_seed

DATA_DIR = Path("/path/to/your/data")
COUNTS_LAYER = None
DEVICE = "auto"
VERBOSE = True
MAX_ITER = 20000
set_seed(1, deterministic=True)

K = 12  # Choose the component number for your data.
TRUTH_CSV = None  # Optional: DATA_DIR / "fractions.csv"


## 2. Align the target and reference feature panels

Reference cell types determine `K`. Keep reference values as raw per-cell means for the cell-fraction
conversion. Features are intersected and ordered by the target; empty intersections fail explicitly.


In [ ]:
adata = ad.read_h5ad(DATA_DIR / "counts.h5ad")
reference_adata = ad.read_h5ad(DATA_DIR / "reference.h5ad")
genes = adata.var_names.intersection(reference_adata.var_names, sort=False)
if not len(genes):
    raise ValueError("The target and reference have no shared feature names")
adata = adata[:, genes].copy()
x = adata.to_df(layer=COUNTS_LAYER)
reference = reference_adata[:, genes].to_df()
xy = np.asarray(adata.obsm["spatial"])
sections = adata.obs["section_id"].to_numpy() if "section_id" in adata.obs else np.repeat("section_1", len(x))
K = len(reference)
display(reference.iloc[:5, :5])


## 3. Fit weights and convert to cell fractions


In [ ]:
from spDecon.spatial import spatial_graph
from spDecon.reference_spatial import fit as fit_fixed_reference
from spDecon.reference import cell_fractions
i, j, w, _ = spatial_graph(xy, sections)
p, b, history, diagnostics = fit_fixed_reference(
    x, reference, i, j, w, 0., tolerance=1e-10, max_steps=100000, verbose=VERBOSE)
fractions = pd.DataFrame(cell_fractions(p, reference.sum(axis=1).to_numpy()),
                         index=x.index, columns=reference.index)
display(diagnostics)
display(fractions.head())


## 4. Plot and export the estimated fractions


In [ ]:
fig, axes = plt.subplots(int(np.ceil(K/4)), 4, figsize=(12, 3*int(np.ceil(K/4))), squeeze=False)
for k, ax in enumerate(axes.ravel()):
    if k < K:
        art = spatial(ax, xy, fractions.iloc[:, k].to_numpy(), str(fractions.columns[k]), size=10)
    else:
        ax.axis("off")
fig.colorbar(art, ax=axes, label="Estimated cell fraction", shrink=.6)
save_figure(fig, "reference_based_fractions")
out = ROOT / "outputs/reference_based"
out.mkdir(parents=True, exist_ok=True)
fractions.to_csv(out / "cell_fractions.csv")


## 5. Optional evaluation

The optional fraction CSV must use the same cell-type names as the reference. If your annotations
use broader groups, aggregate the corresponding columns explicitly before evaluating them.


In [ ]:
if TRUTH_CSV is not None:
    truth = pd.read_csv(TRUTH_CSV, index_col=0)
    predicted = fractions.loc[truth.index, truth.columns]
    scores = pd.DataFrame({"type": truth.columns,
        "PCC": np.diag(correlation(predicted.to_numpy(), truth.to_numpy()))})
    display(save_table(scores, "reference_based_pcc"))
    print("RMSE:", np.sqrt(np.mean((predicted.to_numpy()-truth.to_numpy())**2)))
else:
    print("No fraction labels supplied; estimated fractions have been exported.")
